In [31]:
import pandas as pd
import os
from pathlib import Path 

# **reusable functions**

In [32]:
def print_line(num):
    print('=' * num)
    
def print_txt(text, num=50):
    print_line(num)
    print(text)
    print_line(num)

In [33]:
def check_the_order(column):
    is_ordered_old_new = column.is_monotonic_increasing
    is_ordered_new_old = column.is_monotonic_decreasing
    return is_ordered_old_new, is_ordered_new_old

# **Import data**

In [34]:
# RAW_DATA_DIR = '/kaggle/input/datasets/karapy/azure-1/dataset-electric.csv'

RAW_DATA_DIR = Path('../data/raw/dataset-electric.csv')
SAVE_DATA_DIR = Path('../data/processed')

df = pd.read_csv(RAW_DATA_DIR)

# **Cleaning**

In [35]:


# df['Datetime'] = df['Datetime'].str.strip()
df['Datetime'] = pd.to_datetime(df['Datetime'])

In [36]:
print_txt('check if the Datetime column is ordered correctly')

is_ordered_old_new, is_ordered_new_old = check_the_order(df['Datetime'])

print(f'Ascending order  : {is_ordered_old_new}')
print(f'Descending order : {is_ordered_new_old}') 

check if the Datetime column is ordered correctly
Ascending order  : False
Descending order : False


In [37]:
print_txt('check the rows with wrong order ')

df.loc[df['Datetime'].diff() < pd.Timedelta(0)]

check the rows with wrong order 


,Datetime,PJME_MW
24,2002-12-30 01:00:00,27526.0
48,2002-12-29 01:00:00,27084.0
72,2002-12-28 01:00:00,29344.0
96,2002-12-27 01:00:00,28445.0
120,2002-12-26 01:00:00,26705.0
...,...,...
145246,2018-01-05 01:00:00,39667.0
145270,2018-01-04 01:00:00,34494.0
145294,2018-01-03 01:00:00,38011.0
145318,2018-01-02 01:00:00,37679.0


In [38]:
print_txt('sort the datetime column ascending')
df = df.sort_values(by='Datetime', ascending=True)

print('done')

sort the datetime column ascending
done


In [39]:
print_txt('Check if the order is fixed after sort_values')

is_ordered_old_new, _ = check_the_order(df['Datetime'])
print(f'Ascending order  : {is_ordered_old_new}')

Check if the order is fixed after sort_values
Ascending order  : True


In [40]:
min_data = df['Datetime'].min()
max_data = df['Datetime'].max()
(min_data,max_data)

(Timestamp('2002-01-01 01:00:00'), Timestamp('2018-08-03 00:00:00'))

In [41]:
mask = df['Datetime'].diff() > pd.Timedelta(hours=1)


# make a copy of the Datetime

gap_info = df.loc[mask, ['Datetime']].copy()


gap_info['gap'] = df['Datetime'].diff().loc[mask]

gap_info['missing_hours'] = (gap_info['gap'] / pd.Timedelta(hours=1) - 1)

display(gap_info)

,Datetime,gap,missing_hours
6433,2002-04-07 04:00:00,0 days 02:00:00,1.0
1561,2002-10-27 03:00:00,0 days 02:00:00,1.0
15215,2003-04-06 04:00:00,0 days 02:00:00,1.0
10343,2003-10-26 03:00:00,0 days 02:00:00,1.0
24021,2004-04-04 04:00:00,0 days 02:00:00,1.0
18981,2004-10-31 03:00:00,0 days 02:00:00,1.0
32827,2005-04-03 04:00:00,0 days 02:00:00,1.0
27787,2005-10-30 03:00:00,0 days 02:00:00,1.0
41609,2006-04-02 04:00:00,0 days 02:00:00,1.0
36569,2006-10-29 03:00:00,0 days 02:00:00,1.0


In [42]:
df.loc[df['Datetime'].diff() > pd.Timedelta(hours=1)]

,Datetime,PJME_MW
6433,2002-04-07 04:00:00,24487.0
1561,2002-10-27 03:00:00,20605.0
15215,2003-04-06 04:00:00,22902.0
10343,2003-10-26 03:00:00,20641.0
24021,2004-04-04 04:00:00,23208.0
18981,2004-10-31 03:00:00,20585.0
32827,2005-04-03 04:00:00,23414.0
27787,2005-10-30 03:00:00,23302.0
41609,2006-04-02 04:00:00,20833.0
36569,2006-10-29 03:00:00,23310.0


In [43]:
print_txt('Investigate consumption anomalies')
print('first 10 rows')
# print_line(20)
display(df['PJME_MW'].sort_values(ascending=False).tail(10))
print_line(20)
print('last 10 rows')
display(df['PJME_MW'].sort_values(ascending=True).tail(10))

Investigate consumption anomalies
first 10 rows


89121    17734.0
89160    17422.0
89120    16688.0
89161    15919.0
89119    15526.0
89114    15390.0
89115    14955.0
89118    14821.0
89116    14586.0
89117    14544.0
Name: PJME_MW, dtype: float64

last 10 rows


82769    61532.0
82771    61608.0
38696    61610.0
38693    61641.0
38719    61643.0
82770    61646.0
38670    61770.0
38671    61796.0
38694    61909.0
38695    62009.0
Name: PJME_MW, dtype: float64

In [46]:
SAVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [48]:
df.to_csv(Path.joinpath(SAVE_DATA_DIR, 'cleaned_dataset-electric.csv'), index=False)